# Experiment: Knowledge Graph Q&A Sandbox

Objective:
- Load or refresh Neo4j graph data from already-ingested documents.
- Run Q&A against `/api/v1/ask` and inspect source grounding.

Success criteria:
- Graph indexing reaches zero pending documents (or reports a concrete blocker).
- At least one Q&A run returns an answer with cited sources.


In [ ]:
# Setup: API helpers and strict error handling
from __future__ import annotations

import os
import time
import textwrap
from pprint import pprint

try:
    import requests
except ImportError as exc:
    raise ImportError("This notebook requires `requests`. Install with: pip install requests") from exc

API_BASE_URL = os.getenv("API_BASE_URL", "http:..localhost:8000.api/v1").rstrip("/")
REQUEST_TIMEOUT_SECONDS = int(os.getenv("REQUEST_TIMEOUT_SECONDS", "30"))
INDEX_POLL_INTERVAL_SECONDS = int(os.getenv("INDEX_POLL_INTERVAL_SECONDS", "3"))
INDEX_MAX_WAIT_SECONDS = int(os.getenv("INDEX_MAX_WAIT_SECONDS", "180"))
AUTO_INDEX_MISSING = True

def _raise_for_status(response: requests.Response) -> None:
    if response.ok:
        return
    detail = response.text
    try:
        payload = response.json()
        detail = payload.get("detail", payload)
    except Exception:
        pass
    raise RuntimeError(
        f"{response.request.method} {response.url} failed with status {response.status_code}: {detail}"
    )

def api_get(path: str, params: dict | None = None) -> dict:
    response = requests.get(f"{API_BASE_URL}{path}", params=params, timeout=REQUEST_TIMEOUT_SECONDS)
    _raise_for_status(response)
    return response.json()

def api_post(path: str, json_body: dict | None = None) -> dict:
    response = requests.post(f"{API_BASE_URL}{path}", json=json_body or {}, timeout=REQUEST_TIMEOUT_SECONDS)
    _raise_for_status(response)
    return response.json()

print(f"API_BASE_URL={API_BASE_URL}")
print(f"INDEX_MAX_WAIT_SECONDS={INDEX_MAX_WAIT_SECONDS}")


## Plan

- Check backend readiness (`health`, `documents`, graph stats/indexing status).
- Enqueue indexing for missing documents and poll until indexing completes.
- Ask a small set of Q&A prompts and inspect confidence + sources.
- Capture quick metrics for repeatable sanity checks.


In [ ]:
# Baseline system snapshot
health = api_get("/health")
documents_payload = api_get("/documents")
graph_stats = api_get("/graph/stats")
index_status = api_get("/graph/indexing/status")

snapshot = {
    "health_status": health.get("status"),
    "service_status": health.get("services", {}),
    "document_count": documents_payload.get("total", 0),
    "graph_documents": graph_stats.get("documents", 0),
    "graph_relationships": graph_stats.get("relationships", 0),
    "pending_documents": index_status.get("pending_documents", 0),
}
pprint(snapshot)


In [ ]:
# Load missing graph data into Neo4j and wait for completion
if AUTO_INDEX_MISSING:
    enqueue_result = api_post("/graph/indexing/index-missing", {})
    print("Enqueue result:")
    pprint(enqueue_result)
else:
    enqueue_result = {"skipped": True}
    print("AUTO_INDEX_MISSING is False; skipping index enqueue.")

start_ts = time.time()
index_poll_history = []

while True:
    current = api_get("/graph/indexing/status")
    pending = int(current.get("pending_documents", 0))
    indexed = int(current.get("indexed_documents", 0))
    total = int(current.get("total_documents", 0))
    elapsed = round(time.time() - start_ts, 1)

    index_poll_history.append(
        {
            "elapsed_seconds": elapsed,
            "indexed_documents": indexed,
            "total_documents": total,
            "pending_documents": pending,
        }
    )

    print(f"t={elapsed:>5}s | indexed={indexed}/{total} | pending={pending}")

    if pending == 0:
        final_index_status = current
        break

    if elapsed >= INDEX_MAX_WAIT_SECONDS:
        raise TimeoutError(
            "Graph indexing did not finish before timeout. Check backend logs and confirm Celery worker is running."
        )

    time.sleep(INDEX_POLL_INTERVAL_SECONDS)

print("\nFinal indexing status:")
pprint(final_index_status)


## Q&A Runs

Edit `TEST_QUESTIONS` to match your use case. The cell below calls `/api/v1/ask` directly.


In [ ]:
def ask_question(question: str, document_ids: list[str] | None = None, n_context: int = 5) -> dict:
    payload = {
        "question": question,
        "n_context": n_context,
        "document_ids": document_ids if document_ids is not None else [],
    }
    return api_post("/ask", payload)

TEST_QUESTIONS = [
    "What are the main entities and relationships represented in these documents?",
    "What claim, policy, or invoice identifiers are mentioned and how are they connected?",
]

qa_runs: list[dict] = []

for question in TEST_QUESTIONS:
    result = ask_question(question=question, n_context=5)
    qa_runs.append({"question": question, "result": result})

    print(f"Q: {question}")
    print("A:", textwrap.shorten(result.get("answer", ""), width=220, placeholder=" ..."))
    print("confidence:", result.get("confidence"))
    print("sources:", len(result.get("sources", [])))
    if result.get("sources"):
        top_source = result["sources"][0]
        print("top source:", top_source.get("filename"), "chunk", top_source.get("chunk_index"))
    print("-" * 80)

qa_runs[0] if qa_runs else {}


In [ ]:
# Optional retrieval check: inspect semantic search results directly
SEARCH_QUERY = TEST_QUESTIONS[0]
search_result = api_get("/search", params={"q": SEARCH_QUERY, "n_results": 5})
print("search total:", search_result.get("total", 0))
for row in search_result.get("results", [])[:3]:
    print(
        row.get("filename"),
        "chunk",
        row.get("chunk_index"),
        "similarity",
        row.get("similarity"),
    )


In [ ]:
# Summarize Q&A metrics for quick regression checks
qa_summary = []
for run in qa_runs:
    result = run["result"]
    similarities = [float(source.get("similarity", 0.0)) for source in result.get("sources", [])]
    qa_summary.append(
        {
            "question": run["question"],
            "confidence": float(result.get("confidence", 0.0)),
            "source_count": len(result.get("sources", [])),
            "top_similarity": max(similarities) if similarities else 0.0,
            "answer_chars": len(result.get("answer", "")),
        }
    )

pprint(qa_summary)


## Results

- Capture what looked correct vs. suspicious in the answers.
- Note if confidence tracks answer quality in your domain.
- Record any retrieval gaps (missing entities, weak sources, wrong files).

## Next steps

- Replace `TEST_QUESTIONS` with your acceptance questions.
- Add assertions (for example, minimum source count or confidence threshold) for automated checks.
- Optional: pass `document_ids` in `ask_question(...)` to run scoped Q&A for specific docs.


In [ ]:
from uu_backend.services.qa_service import QAService

from dotenv import load_dotenv
load_dotenv()
qa = QAService()


from neo4j import GraphDatabase
from neo4j_graphrag.embeddings.openai import OpenAIEmbeddings
from neo4j_graphrag.generation import GraphRAG
from neo4j_graphrag.indexes import create_vector_index
from neo4j_graphrag.llm.openai_llm import OpenAILLM
from neo4j_graphrag.retrievers import VectorCypherRetriever
from neo4j_graphrag.types import RetrieverResultItem


qa._neo4j_driver = GraphDatabase.driver(
            qa.settings.neo4j_uri,
            auth=(qa.settings.neo4j_user, qa.settings.neo4j_password),
        )
        

embedder = OpenAIEmbeddings(
            model=qa.settings.graphrag_embedding_model,
            api_key=qa.settings.openai_api_key,
)



In [ ]:
qa._retriever.search(query_text='auto claim')

In [ ]:

qa._retriever = VectorCypherRetriever(
    driver=qa._neo4j_driver,
    index_name=qa.settings.graphrag_vector_index_name,
    retrieval_query=qa._VECTOR_RETRIEVAL_QUERY,
    embedder=embedder,
    result_formatter=qa._record_to_retriever_item,
    neo4j_database=qa.settings.neo4j_database,
)

In [ ]:

llm = OpenAILLM(
    model_name=qa.model,
    api_key=qa.settings.openai_api_key,
)

qa._graphrag = GraphRAG(retriever=qa._retriever, llm=llm)

In [ ]:
qa._graphrag.search(
            query_text='What are the main entities and relationships represented in these documents?',
            retriever_config={
                "top_k": 3,
                "query_params": {"document_ids":  []},
            },
            return_context=True,
        )